In [2]:
import os, time, random, math
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 128
EPOCHS_A = 2   # rápido para CPU
EPOCHS_B = 2
LR_HEAD = 1e-3
LR_FT = 3e-4
SEED = 42
torch.manual_seed(SEED); random.seed(SEED)

# 1) Datos: CIFAR-10 32x32, normalizado
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2470, 0.2435, 0.2616))
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2470, 0.2435, 0.2616))
])

train_full = datasets.CIFAR10(root="data", train=True, download=True, transform=transform_train)
test = datasets.CIFAR10(root="data", train=False, download=True, transform=transform_test)

# Subconjuntos para ir rápido
idx_train = list(range(10000))   # 10k
idx_val = list(range(2000))      # 2k (del train para validar rápido)
train = Subset(train_full, idx_train)
val = Subset(train_full, idx_val)

dl_train = DataLoader(train, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, num_workers=2)
dl_val = DataLoader(val, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

def build_model(num_classes=10, pretrained=True):
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT if pretrained else None)
    in_feats = model.fc.in_features
    model.fc = nn.Linear(in_feats, num_classes)
    return model

def eval_model(model, loader):
    model.eval()
    correct = 0; total = 0; loss_sum = 0.0
    ce = nn.CrossEntropyLoss()
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits = model(x)
            loss = ce(logits, y)
            loss_sum += loss.item()
            pred = logits.argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.size(0)
    return loss_sum/len(loader), correct/total


In [3]:
# ------------------------ Experimento A: Feature Extraction ------------------------
def experiment_A_feature_extraction():
    model = build_model().to(DEVICE)

    # congelar todo excepto la capa final
    for p in model.parameters():
        p.requires_grad = False
    for p in model.fc.parameters():
        p.requires_grad = True

    opt = optim.Adam(model.fc.parameters(), lr=LR_HEAD)
    ce = nn.CrossEntropyLoss()

    print("==> EXPERIMENTO A: Feature Extraction (solo cabeza)")
    for epoch in range(1, EPOCHS_A+1):
        model.train()
        running = 0.0
        for x, y in dl_train:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            logits = model(x)
            loss = ce(logits, y)
            loss.backward()
            opt.step()
            running += loss.item()
        val_loss, val_acc = eval_model(model, dl_val)
        print(f"[A] epoch {epoch}/{EPOCHS_A} train_loss={running/len(dl_train):.3f} "
              f"val_loss={val_loss:.3f} val_acc={val_acc:.3f}")
    return model


In [4]:
# ------------------------ Experimento B: Fine-Tuning parcial -----------------------
def experiment_B_finetune_partial(model=None):
    # si no se pasa modelo, crear otro desde cero para comparación justa
    model = build_model().to(DEVICE) if model is None else model

    # descongelar último bloque (layer4) y la cabeza
    for p in model.parameters():
        p.requires_grad = False
    for p in model.layer4.parameters():
        p.requires_grad = True
    for p in model.fc.parameters():
        p.requires_grad = True

    # LR menor para capas preentrenadas, LR_HEAD para la cabeza
    params = [
        {"params": model.layer4.parameters(), "lr": LR_FT},
        {"params": model.fc.parameters(), "lr": LR_HEAD},
    ]
    opt = optim.Adam(params, lr=LR_FT)
    ce = nn.CrossEntropyLoss()

    print("==> EXPERIMENTO B: Fine-Tuning parcial (layer4 + cabeza)")
    for epoch in range(1, EPOCHS_B+1):
        model.train()
        running = 0.0
        for x, y in dl_train:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            logits = model(x)
            loss = ce(logits, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()
            running += loss.item()
        val_loss, val_acc = eval_model(model, dl_val)
        print(f"[B] epoch {epoch}/{EPOCHS_B} train_loss={running/len(dl_train):.3f} "
              f"val_loss={val_loss:.3f} val_acc={val_acc:.3f}")
    return model

if __name__ == "__main__":
    mA = experiment_A_feature_extraction()
    mB = experiment_B_finetune_partial()
    print("Listo. Compara val_acc de A vs B y tiempos de ejecución.")


==> EXPERIMENTO A: Feature Extraction (solo cabeza)
[A] epoch 1/2 train_loss=2.055 val_loss=1.763 val_acc=0.386
[A] epoch 2/2 train_loss=1.719 val_loss=1.616 val_acc=0.449
==> EXPERIMENTO B: Fine-Tuning parcial (layer4 + cabeza)
[B] epoch 1/2 train_loss=1.521 val_loss=0.901 val_acc=0.698
[B] epoch 2/2 train_loss=0.979 val_loss=0.603 val_acc=0.793
Listo. Compara val_acc de A vs B y tiempos de ejecución.
